# Multi-Agent Coordination using Tabular Q-Learning

## Objective

The objective of this project is to implement a multi-agent reinforcement learning system using Tabular Q-Learning.

Two autonomous agents (Type A and Type B) operate in the same 5×5 Grid World. Each agent learns independently to collect a sample and return it to its own base while avoiding collisions and adapting to the changing lake condition.

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import time

from IPython.display import clear_output

## Project Constants

In [2]:
# Grid Size
GRID_SIZE = 5

# Total Episodes
EPISODES = 20000

# Maximum Steps per Episode
MAX_STEPS = 100

## Q-Learning Hyperparameters

In [3]:
# Learning Rate
ALPHA = 0.1

# Discount Factor
GAMMA = 0.95

# Exploration Rate
EPSILON = 1.0

# Exploration Decay
EPSILON_DECAY = 0.995

# Minimum Exploration
MIN_EPSILON = 0.01

## Fixed Locations

In [4]:
# Landing Pad X (West)
X_ROW = 2
X_COL = 0

# Landing Pad Y (North)
Y_ROW = 0
Y_COL = 2

# Sampling Site U (East)
U_ROW = 2
U_COL = 4

# Sampling Site V (South)
V_ROW = 4
V_COL = 2

## Lake Configuration

In [5]:
# Lake Position (Center)

LAKE_ROW = 2
LAKE_COL = 2

# Lake State

DRY = 0
FLOODED = 1

# Initial Lake State

lake_state = DRY

# Probability that the lake changes state
LAKE_FLIP_PROBABILITY = 0.2

## Action Space

In [6]:
ACTIONS = [
    "UP",
    "DOWN",
    "LEFT",
    "RIGHT",
    "WAIT"
]

TOTAL_ACTIONS = len(ACTIONS)

print(TOTAL_ACTIONS)

5


## State Representation

In [7]:
TOTAL_STATES = (
    GRID_SIZE *
    GRID_SIZE *
    2 *
    2
)

print("Total States:", TOTAL_STATES)

Total States: 100


## State Encoding Function

In [8]:
def state_to_index(
    row,
    col,
    has_sample,
    lake_state
):

    return (
        (
            (
                row * GRID_SIZE
                + col
            )
            * 2
            + int(has_sample)
        )
        * 2
        + lake_state
    )

## Initialize Q-Tables

In [9]:
q_table_A = np.zeros(
    (
        TOTAL_STATES,
        TOTAL_ACTIONS
    )
)

q_table_B = np.zeros(
    (
        TOTAL_STATES,
        TOTAL_ACTIONS
    )
)

print("Q-Table A:", q_table_A.shape)
print("Q-Table B:", q_table_B.shape)

Q-Table A: (100, 5)
Q-Table B: (100, 5)


## Environment Reset Function

In [10]:
def reset_environment():
    """
    Reset the environment before every episode.
    """

    # Agent A starts from X
    agent_a_row = X_ROW
    agent_a_col = X_COL

    # Agent B starts from Y
    agent_b_row = Y_ROW
    agent_b_col = Y_COL

    # Initially no agent has a sample
    agent_a_has_sample = False
    agent_b_has_sample = False

    # Randomly initialize lake state
    lake_state = random.choice([DRY, FLOODED])

    return (
        agent_a_row,
        agent_a_col,
        agent_b_row,
        agent_b_col,
        agent_a_has_sample,
        agent_b_has_sample,
        lake_state
    )

## Test Environment Reset

In [19]:
(
    agent_a_row,
    agent_a_col,
    agent_b_row,
    agent_b_col,
    agent_a_has_sample,
    agent_b_has_sample,
    lake_state
) = reset_environment()

print("Agent A :", (agent_a_row, agent_a_col))
print("Agent B :", (agent_b_row, agent_b_col))

print("Agent A Sample :", agent_a_has_sample)
print("Agent B Sample :", agent_b_has_sample)

print("Lake State :", "DRY" if lake_state == DRY else "FLOODED")

Agent A : (2, 0)
Agent B : (0, 2)
Agent A Sample : False
Agent B Sample : False
Lake State : DRY


## Lake Flip Function

In [20]:
def update_lake_state(current_state):
    """
    Update lake state with probability p.
    """

    if random.random() < LAKE_FLIP_PROBABILITY:

        if current_state == DRY:
            return FLOODED

        return DRY

    return current_state

## Test Lake Flip

In [22]:
lake = DRY

for i in range(10):

    lake = update_lake_state(lake)

    print(
        "Step",
        i + 1,
        ":",
        "DRY" if lake == DRY else "FLOODED"
    )

Step 1 : FLOODED
Step 2 : FLOODED
Step 3 : FLOODED
Step 4 : FLOODED
Step 5 : DRY
Step 6 : DRY
Step 7 : FLOODED
Step 8 : FLOODED
Step 9 : DRY
Step 10 : DRY


## Pickup Logic

In [23]:
def pickup_sample(
    row,
    col,
    has_sample,
    target_row,
    target_col
):

    if (
        row == target_row
        and
        col == target_col
    ):
        has_sample = True

    return has_sample

## Delivery Logic

In [24]:
def deliver_sample(
    row,
    col,
    has_sample,
    home_row,
    home_col
):

    if (
        row == home_row
        and
        col == home_col
        and
        has_sample
    ):

        has_sample = False

        return True, has_sample

    return False, has_sample

## Environment Helper Functions

In [25]:
def is_lake_cell(row, col):

    return (
        row == LAKE_ROW
        and
        col == LAKE_COL
    )

## Test Helper Function

In [26]:
print(is_lake_cell(2, 2))
print(is_lake_cell(1, 1))

True
False
